# AlphaKraken REST API Demo

Demonstrating how to retrieve data from the REST API.

In [ ]:
import pandas as pd
import requests

BASE_URL = "http://localhost:8090"
TIMEOUT = 10

In [ ]:
def get(  # noqa: PLR0913
    *,
    instrument_id: str | None = None,
    name_contains: str | None = None,
    project_id: str | None = None,
    date_from: str | None = None,
    date_to: str | None = None,
    limit: int | None = None,
    offset: int | None = None,
) -> dict:
    """Get data from the REST API."""
    params = {k: v for k, v in locals().items() if v is not None}
    return requests.get(f"{BASE_URL}/raw_files/", params=params, timeout=TIMEOUT).json()

In [ ]:
def to_dataframe(data: dict) -> pd.DataFrame:
    """Convert API response to a DataFrame, expanding metrics into columns prefixed by type."""
    rows = []
    for raw_file in data["data"]:
        row = {k: v for k, v in raw_file.items() if k != "metrics"}
        for m in raw_file.get("metrics", []):
            prefix = m.get("type", "metrics")
            for k, v in m.items():
                if k != "type":
                    row[f"{prefix}:{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)

## Get recent raw files

In [ ]:
data = get(limit=5)
print(f"Total: {data['total']}, returned: {len(data['data'])}")
data["data"][0]

## Filter by instrument and date range

In [ ]:
get(instrument_id="test1", date_from="2025-01-01", date_to="2025-12-31", limit=3)

## Search by name substring

In [ ]:
get(name_contains="hela", limit=3)

## Pagination

In [ ]:
page1 = get(limit=2, offset=0)
page2 = get(limit=2, offset=2)

print(f"Total: {page1['total']}")
print(f"Page 1 IDs: {[r['id'] for r in page1['data']]}")
print(f"Page 2 IDs: {[r['id'] for r in page2['data']]}")

## Load into DataFrame

In [ ]:
df = to_dataframe(get(limit=50, instrument_id="stellar1"))
df.head()

## Projects with their settings assignments

In [ ]:
requests.get(f"{BASE_URL}/projects/", params={"limit": 3}, timeout=TIMEOUT).json()

## Settings

In [ ]:
requests.get(f"{BASE_URL}/settings/fallback_v1", timeout=TIMEOUT).json()

## Writes (need the API key)

The writes and `GET /users/` need the header `X-API-Key` with the value of `REST_API_KEY` of the deployment.
The cells below change the database: run them against a local or sandbox instance.

In [ ]:
API_KEY = "local-api-key"  # REST_API_KEY in envs/local.env
HEADERS = {"X-API-Key": API_KEY}


def call(method: str, path: str, **kwargs) -> dict:
    """Call a keyed endpoint and return its response, empty if it has none; raise with the error details of the API if it fails."""
    response = requests.request(
        method, f"{BASE_URL}{path}", headers=HEADERS, timeout=TIMEOUT, **kwargs
    )
    if not response.ok:
        raise RuntimeError(f"{response.status_code}: {response.json()['detail']}")
    return response.json() if response.content else {}

### Possible owners of projects and settings

In [ ]:
call("GET", "/users/")

### Create a project and settings, and assign the settings

In [ ]:
project = call(
    "POST", "/projects/", json={"project_id": "DEMO1", "name": "Demo project"}
)
project

In [ ]:
settings = call(
    "POST",
    "/settings/",
    json={
        "name": "demo",
        "software_type": "msqc",
        "software": "msqc-1.0.0",
        "runner_name": "slurm",  # a name in alphakraken.yaml `runners`
        "metrics_type": "msqc",
    },
)
settings["id"]

In [ ]:
assignment = call(
    "POST",
    "/assignments/",
    json={
        "project_id": project["id"],
        "settings_id": settings["id"],
        "scopes": ["*"],  # all instruments
        "excluded_scopes": [],
        "raw_file_id_filter": [],
        "raw_file_id_exclude_filter": [],
    },
)
assignment

### Errors

An invalid request gives `422` with all violations, a wrong or missing key `401`.

In [ ]:
try:
    call("POST", "/projects/", json={"project_id": "demo 1", "nam": "typo"})
except RuntimeError as e:
    print(e)

### Clean up: unassign, then delete the settings and the project

In [ ]:
call("DELETE", f"/assignments/{assignment['id']}")
call("DELETE", f"/settings/{settings['id']}")
call("DELETE", f"/projects/{project['id']}")